# Monte-Carlo-Schätzung von $\pi$

Wir ziehen gleichverteilte Punkte aus einem Quadrat und zählen, welcher Anteil in einem Kreis mit Radius $R$ liegt. Aus dem Flächenverhältnis folgt

$$p=\frac{A_\mathrm{Kreis}}{A_\mathrm{Quadrat}}=\frac{\pi R^2}{4a^2},$$

wobei $a$ die halbe Seitenlänge des Quadrats ist. Für den gemessenen Trefferanteil $\hat p$ erhalten wir daher den Schätzer

$$\hat\pi=4\left(\frac{a}{R}\right)^2\hat p.$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
R = 1.0

def estimate_pi(n_samples, size_ratio, rng):
    """Schätze pi in einem Quadrat mit a/R = size_ratio."""
    a = size_ratio * R
    points = rng.uniform(-a, a, size=(n_samples, 2))
    inside = np.sum(points**2, axis=1) <= R**2
    pi_estimate = 4 * size_ratio**2 * inside.mean()
    return pi_estimate, points, inside

## Ein festes Quadrat: Treffer als Bernoulli-Variable

Zunächst verwenden wir das eng anliegende Quadrat mit $a/R=1$. Jedem Zufallspunkt ordnen wir die Indikatorvariable

$$X_i=\begin{cases}1,&\text{Punkt liegt im Kreis},\\0,&\text{Punkt liegt außerhalb.}\end{cases}$$

zu. Die Trefferwahrscheinlichkeit ist der Erwartungswert $p=\langle X_i\rangle$. Weil $X_i$ nur die Werte 0 und 1 annimmt, gilt $X_i^2=X_i$ und damit besonders einfach

$$\langle X_i^2\rangle=\langle X_i\rangle=p,\qquad \operatorname{Var}(X_i)=p-p^2=p(1-p).$$

Der gemessene Trefferanteil $\hat p=N^{-1}\sum_i X_i$ ist ein Mittelwert über $N$ unabhängige Variablen. Deshalb ist $\operatorname{Var}(\hat p)=p(1-p)/N$. Für $a/R=1$ gilt $\hat\pi=4\hat p$ und folglich

$$\operatorname{Var}(\hat\pi)=\frac{16p(1-p)}{N}=\frac{\pi(4-\pi)}{N}.$$

In [ ]:
ratio = 1.0
n_visual = 2000
pi_hat, points, inside = estimate_pi(n_visual, ratio, rng)
angle = np.linspace(0, 2 * np.pi, 400)

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(points[~inside, 0], points[~inside, 1], s=5, alpha=0.45,
           color='tab:red', label='$X_i=0$')
ax.scatter(points[inside, 0], points[inside, 1], s=5, alpha=0.45,
           color='tab:blue', label='$X_i=1$')
ax.plot(R * np.cos(angle), R * np.sin(angle), color='black', linewidth=1.5)
ax.set(xlim=(-1, 1), ylim=(-1, 1), aspect='equal', xlabel='$x$', ylabel='$y$',
       title=fr'$N={n_visual}$, $\hat{{\pi}}={pi_hat:.4f}$')
ax.legend(loc='upper right', markerscale=2)
plt.tight_layout()
plt.show()

## Viele Experimente und der zentrale Grenzwertsatz

Nun wiederholen wir das gesamte Experiment viele Male. Die linke Spalte zeigt die Folge der erhaltenen $\hat\pi$-Werte. In der rechten Spalte wird deren normiertes Histogramm mit der theoretisch vorhergesagten Gaußverteilung verglichen. Dabei werden keine Parameter angepasst: Es gelten $\mu=\pi$ und $\sigma=\sqrt{\pi(4-\pi)/N}$.

Der zentrale Grenzwertsatz besagt, dass der Mittelwert $\hat p$ für großes $N$ näherungsweise gaußverteilt ist. Bei $N=1000$ sollte das Histogramm daher schmaler und die Gaußnäherung deutlicher sein als bei $N=100$. Die Breite nimmt wie $1/\sqrt{N}$ ab.

In [ ]:
sample_sizes = [100, 1000]
n_experiments = 10000
histories = {}

for n_samples in sample_sizes:
    histories[n_samples] = np.array([
        estimate_pi(n_samples, size_ratio=1.0, rng=rng)[0]
        for _ in range(n_experiments)
    ])

fig, axes = plt.subplots(2, 2, figsize=(12, 7))

for row, n_samples in enumerate(sample_sizes):
    estimates = histories[n_samples]
    sigma_theory = np.sqrt(np.pi * (4 - np.pi) / n_samples)

    ax_history = axes[row, 0]
    ax_history.plot(estimates, linewidth=0.6, alpha=0.75)
    ax_history.axhline(np.pi, color='black', linestyle='--', label=r'$\pi$')
    ax_history.fill_between(
        [0, n_experiments - 1], np.pi - sigma_theory, np.pi + sigma_theory,
        color='tab:orange', alpha=0.2, label=r'$\pi\pm\sigma_\mathrm{theo}$'
    )
    ax_history.set(xlabel='Nummer des Experiments', ylabel=r'$\hat{\pi}$',
                   title=f'Folge der Schätzwerte, N = {n_samples}')
    ax_history.legend(loc='upper right')

    ax_hist = axes[row, 1]
    ax_hist.hist(estimates, bins=35, density=True, alpha=0.65,
                 color='tab:blue', label='Monte-Carlo-Daten')
    x = np.linspace(np.pi - 4 * sigma_theory, np.pi + 4 * sigma_theory, 500)
    gaussian_theory = np.exp(-0.5 * ((x - np.pi) / sigma_theory)**2) / (
        np.sqrt(2 * np.pi) * sigma_theory
    )
    ax_hist.plot(x, gaussian_theory, color='tab:red', linewidth=2,
                 label=fr'Theorie: $\mu=\pi$, $\sigma={sigma_theory:.4f}$')
    ax_hist.axvline(np.pi, color='black', linestyle='--', label=r'$\pi$')
    ax_hist.set(xlabel=r'$\hat{\pi}$', ylabel='Wahrscheinlichkeitsdichte',
                title=f'Verteilung über {n_experiments} Experimente')
    ax_hist.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(' N      Mittelwert    Std. (Empirie)    Std. (Theorie)')
for n_samples in sample_sizes:
    estimates = histories[n_samples]
    sigma_theory = np.sqrt(np.pi * (4 - np.pi) / n_samples)
    print(f'{n_samples:4d}    {estimates.mean():10.5f}    '
          f'{estimates.std(ddof=1):10.5f}    {sigma_theory:14.5f}')

## Vergleich unterschiedlich großer Quadrate

Jetzt bleibt der Kreis unverändert, während das umschließende Quadrat größer wird. Im kleinsten zulässigen Quadrat ($a/R=1$) ist der Trefferanteil am größten. Ein größeres Quadrat enthält mehr Punkte, die weit außerhalb des Kreises liegen und für die Schätzung wenig Information liefern.

In [ ]:
size_ratios = [1.0, 1.25, 1.5, 2.0]
n_samples = 4000

fig, axes = plt.subplots(1, len(size_ratios), figsize=(14, 3.5))
angle = np.linspace(0, 2 * np.pi, 400)

for ax, ratio in zip(axes, size_ratios):
    pi_hat, points, inside = estimate_pi(n_samples, ratio, rng)
    ax.scatter(points[~inside, 0], points[~inside, 1], s=2, alpha=0.35, color='tab:red')
    ax.scatter(points[inside, 0], points[inside, 1], s=2, alpha=0.35, color='tab:blue')
    ax.plot(R * np.cos(angle), R * np.sin(angle), color='black', linewidth=1.5)
    ax.set(xlim=(-ratio, ratio), ylim=(-ratio, ratio), aspect='equal',
           title=fr'$a/R={ratio}$' + '\n' + fr'$\hat{{\pi}}={pi_hat:.4f}$')
    ax.set_xlabel('$x$')

axes[0].set_ylabel('$y$')
fig.suptitle(f'Jeweils {n_samples} gleichverteilte Punkte', y=1.04)
plt.tight_layout()
plt.show()

## Abhängigkeit der Varianz von der Quadratgröße

Mit $p=\pi R^2/(4a^2)$ und der oben hergeleiteten Bernoulli-Varianz ergibt sich für $N$ unabhängige Punkte

$$\operatorname{Var}(\hat\pi)=\frac{\pi\left[4(a/R)^2-\pi\right]}{N}.$$

Die relative statistische Unsicherheit ist daher

$$\frac{\sqrt{\operatorname{Var}(\hat\pi)}}{\langle\hat\pi\rangle}=\sqrt{\frac{4(a/R)^2-\pi}{\pi N}}.$$

Die Varianz wächst somit mit der Quadratgröße. Das folgende Experiment wiederholt jede Schätzung oft und vergleicht die beobachtete Standardabweichung mit der theoretischen Vorhersage.

In [ ]:
n_samples = 2000
n_repetitions = 500
results = {}

for ratio in size_ratios:
    estimates = np.array([
        estimate_pi(n_samples, ratio, rng)[0]
        for _ in range(n_repetitions)
    ])
    results[ratio] = estimates

empirical_std = np.array([results[r].std(ddof=1) for r in size_ratios])
theoretical_std = np.sqrt(
    np.pi * (4 * np.asarray(size_ratios)**2 - np.pi) / n_samples
)

print(' a/R    Mittelwert    empirische Std.    theoretische Std.')
for ratio, std_emp, std_theory in zip(size_ratios, empirical_std, theoretical_std):
    print(f'{ratio:4.2f}    {results[ratio].mean():9.5f}    {std_emp:15.5f}    {std_theory:16.5f}')

x = np.arange(len(size_ratios))
width = 0.36
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - width/2, empirical_std, width, label='Experiment')
ax.bar(x + width/2, theoretical_std, width, label='Theorie')
ax.set_xticks(x, [str(r) for r in size_ratios])
ax.set_xlabel('Relative halbe Seitenlänge $a/R$')
ax.set_ylabel(r'Standardabweichung von $\hat{\pi}$')
ax.set_title(f'Streuung bei N = {n_samples} Punkten')
ax.legend()
plt.tight_layout()
plt.show()

## Ergebnis

Alle Varianten liefern im Mittel denselben Wert $\pi$: Der Schätzer ist unverzerrt. Das eng anliegende Quadrat mit $a/R=1$ besitzt jedoch die kleinste Varianz und erreicht deshalb bei gleicher Anzahl von Zufallspunkten die höchste Genauigkeit.